**Linear Regression using Vectorization :** </br>
Linear regression is a basic regression model. In my implementation, I use diabetic dataset for regression. In the following section, I import various module like *NumPY*, *pandas*.

In [1]:
import numpy as np
from sklearn.datasets import load_diabetes
import pandas as pd

**Load Dataset :** </br> In this section, I load the dataset from the sklearn and extract the features and label for the corresponding instances. Further, convert those into the vectorized form.

In [2]:
def load_and_preprocess():
    """Loads the sklearn diabetes dataset and prepares matrices as column vectors."""
    # Load dataset: N = 442 examples, d = 10 features
    diabetes = load_diabetes()
    X_raw = diabetes.data  # shape (442, 10)
    y = diabetes.target.reshape(-1, 1)  # shape (442, 1) - Column Vector

    # Add a column of ones to represent the bias term (w0)
    m = X_raw.shape[0]
    X_with_bias = np.c_[np.ones(m), X_raw]  # shape (442, 11)

    # Transpose X to shape (n, m) where n = d + 1
    # This matches the notation: y_hat = X^T * W
    X = X_with_bias.T  # shape (11, 442)
    return X, y, diabetes.feature_names

**Compute Cost :**</br> In this section, I compute the loss of the model using the actual output and the predected output.

In [3]:
def compute_cost(X, y, W):
    """
    Computes cost J = 1/2 * sum((y_i - y_hat_i)^2)
    where y_hat = X^T * W (Result: N x 1 Column Vector)
    """
    y_hat = np.dot(X.T, W)
    error = y - y_hat
    # Calculated as the sum of squared errors as requested
    cost = 0.5 * np.sum(error**2)
    return cost

**Compute gradient :**</br> In the following section, I compute the gradient to update the weights and the bias for optimal solution.

In [4]:
def compute_gradient(X, y, W):
    """
    Computes the vectorized gradient:
    DJ/DW = -X * (y - X^T * W)
    Result shape: (d+1) x 1 Column Vector
    """
    y_hat = np.dot(X.T, W)
    error = y - y_hat
    # Vectorized gradient calculation using the provided formula
    gradient = -np.dot(X, error)
    return gradient

**Apply Gradient Descent :**</br> In this section, we apply the gradent descent to adjust the value of the weights and bias iteratively with a learning rate.

In [5]:
def run_gradient_descent(X, y, alpha=1e-4, max_iters=100000, tol=1e-5):
    """
    Optimizes W using gradient descent with a 5-iteration termination condition.
    """
    n = X.shape[0]  # number of parameters (d + 1)
    # Initialize W as a (d+1) x 1 Column Vector
    W = np.zeros((n, 1))
    cost_history = []

    for i in range(max_iters):
        # 1. Calculate gradient and update weights
        grad = compute_gradient(X, y, W)
        W = W - alpha * grad

        # 2. Record the cost
        current_cost = compute_cost(X, y, W)
        cost_history.append(current_cost)

        # 3. Termination Logic: check change over the previous 5 iterations
        if i >= 5:
            change = abs(cost_history[i-5] - cost_history[i])
            if change < tol:
                print(f"Convergence met at iteration {i}.")
                break

    return W, cost_history

**Driver Code :** </br> In this section, I wrote the driver code which loads the dataset and extract the features and the labels. Further calculate the gradiant and update the weights and bias using the learning rate. Finally show the actual and predicted output for some instances.

In [6]:
# 1. Load data
X, y, feature_names = load_and_preprocess()

# 2. Run Optimization
# alpha is small because the cost is a raw sum, making gradients very large
final_W, history = run_gradient_descent(X, y, alpha=1e-4, max_iters=100000)

# 3. Output Dimensions and Parameters
print(f"Final Bias (w0): {final_W[0][0]:.4f}")
print("Final Weights (w1 - w10):")
for name, weight in zip(feature_names, final_W[1:].flatten()):
    print(f"  {name}: {weight:.4f}")

# 4. Compare Predicted and Target Values (First 10 rows)
# y_hat_final calculated as X^T * W
y_hat_final = np.dot(X.T, final_W)
comparison_df = pd.DataFrame({
    'Target (y)': y.flatten()[:10],
    'Predicted (y_hat)': y_hat_final.flatten()[:10]
})

print("\nPredicted vs Target Value Comparison (First 10 Rows):")
print(comparison_df.to_string(index=False))

Final Bias (w0): 152.1335
Final Weights (w1 - w10):
  age: -6.5830
  sex: -236.6050
  bmi: 529.0263
  bp: 322.1418
  s1: -93.1440
  s2: -89.3219
  s3: -198.3220
  s4: 110.5587
  s5: 483.8417
  s6: 70.5286

Predicted vs Target Value Comparison (First 10 Rows):
 Target (y)  Predicted (y_hat)
      151.0         203.514259
       75.0          70.806169
      141.0         174.257488
      206.0         163.229192
      135.0         128.030627
       97.0         105.672008
      138.0          78.619365
       63.0         122.325583
      110.0         159.297194
      310.0         213.873499
